### 1. Environment Setup & Data Loading
We will install the Kaggle CLI to fetch the commodity prices dataset, and load the local FAO crop storage life dataset.

In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

fao_path = '/content/fao_crop_storage_life_table.csv.xls'

# Let's try to read it as a standard CSV first (since it has .csv in the filename)
try:
    df_fao = pd.read_csv(fao_path)
    print("Successfully loaded FAO table using read_csv!")
    display(df_fao.head())
except Exception as e1:
    print(f"Could not read as CSV: {e1}")
    try:
        # If CSV fails, let's try reading as Excel using different engines
        df_fao = pd.read_excel(fao_path, engine='xlrd')
        print("Successfully loaded FAO table using read_excel with xlrd!")
        display(df_fao.head())
    except Exception as e2:
        print(f"Could not read as Excel with xlrd: {e2}")
        try:
            df_fao = pd.read_excel(fao_path, engine='openpyxl')
            print("Successfully loaded FAO table using read_excel with openpyxl!")
            display(df_fao.head())
        except Exception as e3:
            print(f"Could not read as Excel with openpyxl: {e3}")

Successfully loaded FAO table using read_csv!


,crop,recommended_temperature_c_source_range,recommended_relative_humidity_pct_source_range,approximate_storage_life_days_source_range,temperature_c_min,temperature_c_max,relative_humidity_pct_min,relative_humidity_pct_max,storage_life_days_min,storage_life_days_max,data_type,source,source_url,notes
0,Amaranth,0-2,95-100,10-14,0.0,2.0,95.0,100.0,10.0,14.0,Published reference range; not an observed per...,"FAO, Manual for the preparation and sale of fr...",https://www.fao.org/4/y4893e/y4893e06.htm,Approximate storage life under recommended tem...
1,Apple,-1-4,90-95,30-180,-1.0,4.0,90.0,95.0,30.0,180.0,Published reference range; not an observed per...,"FAO, Manual for the preparation and sale of fr...",https://www.fao.org/4/y4893e/y4893e06.htm,Approximate storage life under recommended tem...
2,Apricot,-0.5-0,90-95,7-21,-0.5,0.0,90.0,95.0,7.0,21.0,Published reference range; not an observed per...,"FAO, Manual for the preparation and sale of fr...",https://www.fao.org/4/y4893e/y4893e06.htm,Approximate storage life under recommended tem...
3,Artichoke,0,95-100,14-21,0.0,0.0,95.0,100.0,14.0,21.0,Published reference range; not an observed per...,"FAO, Manual for the preparation and sale of fr...",https://www.fao.org/4/y4893e/y4893e06.htm,Approximate storage life under recommended tem...
4,Asian pear,1,90-95,150-180,1.0,1.0,90.0,95.0,150.0,180.0,Published reference range; not an observed per...,"FAO, Manual for the preparation and sale of fr...",https://www.fao.org/4/y4893e/y4893e06.htm,Approximate storage life under recommended tem...


### 2. Downloading the Kaggle Dataset
To download the 'Daily Commodity Prices India' dataset from Kaggle, you will need to upload your `kaggle.json` API token to `/content/kaggle.json` or configure your environment variables. Here is the code to download and unzip the dataset.

In [2]:
# Setup Kaggle credentials path if you uploaded kaggle.json to /content/
if os.path.exists('/content/kaggle.json'):
    !mkdir -p ~/.kaggle
    !cp /content/kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

# Download the dataset using Kaggle CLI
!kaggle datasets download -d khandelwalmanas/daily-commodity-prices-india --unzip -p /content/commodity_prices

Dataset URL: https://www.kaggle.com/datasets/khandelwalmanas/daily-commodity-prices-india
License(s): other
100% 1.41G/1.41G [00:15<00:00, 95.4MB/s]



In [5]:
# Inspect files inside the subdirectories
import glob

all_csv_files = glob.glob('/content/commodity_prices/csv/**/*.csv', recursive=True) + glob.glob('/content/commodity_prices/**/*.csv', recursive=True)
print("Found CSV files:", all_csv_files[:10])

if all_csv_files:
    # Load a preview of the first CSV found
    df_prices = pd.read_csv(all_csv_files[0])
    print(f"\nSuccessfully loaded {all_csv_files[0]}!")
    display(df_prices.head())
else:
    print("No CSV files found. Let's list the full contents of the directory:")
    for root, dirs, files in os.walk('/content/commodity_prices'):
        print(f"Root: {root}, Dirs: {dirs}, Files count: {len(files)}")

Found CSV files: ['/content/commodity_prices/csv/2019.csv', '/content/commodity_prices/csv/2001.csv', '/content/commodity_prices/csv/2008.csv', '/content/commodity_prices/csv/2007.csv', '/content/commodity_prices/csv/2010.csv', '/content/commodity_prices/csv/2024.csv', '/content/commodity_prices/csv/2005.csv', '/content/commodity_prices/csv/2015.csv', '/content/commodity_prices/csv/2002.csv', '/content/commodity_prices/csv/2013.csv']

Successfully loaded /content/commodity_prices/csv/2019.csv!


,State,District,Market,Commodity,Variety,Grade,Arrival_Date,Min_Price,Max_Price,Modal_Price,Commodity_Code
0,Andhra Pradesh,Anantapur,Hindupur,Dry Chillies,1st Sort,FAQ,2019-01-01,4000.0,15500.0,9750.0,132
1,Andhra Pradesh,Chittor,Kalikiri,Tomato,Local,FAQ,2019-01-01,400.0,1200.0,800.0,78
2,Andhra Pradesh,Chittor,Mulakalacheruvu,Tomato,Local,FAQ,2019-01-01,220.0,550.0,350.0,78
3,Andhra Pradesh,Chittor,Palamaner,Beetroot,Beetroot,FAQ,2019-01-01,1250.0,2250.0,1750.0,157
4,Andhra Pradesh,Chittor,Palamaner,Bhindi (Ladies Finger),Bhindi,FAQ,2019-01-01,500.0,1000.0,750.0,85


### 3. Data Merging, Feature Engineering & Machine Learning Model
We will join the FAO storage information with the commodity price data. Then, we will calculate price momentum, determine a binary target variable (`1` for **Store**, `0` for **Sell** based on whether prices increase significantly within the crop's storage life window), and train a Random Forest classifier.

In [6]:
# Let's clean the crop names to facilitate matching between datasets
df_fao['crop_clean'] = df_fao['crop'].str.lower().str.strip()

# Let's inspect unique commodities in a subset of the price data to map them
# We will use data from a subset of recent years to make the demo run quickly
recent_files = ['/content/commodity_prices/csv/2019.csv', '/content/commodity_prices/csv/2024.csv']
df_prices_sample = pd.concat([pd.read_csv(f) for f in recent_files if os.path.exists(f)], ignore_index=True)
df_prices_sample['Commodity_clean'] = df_prices_sample['Commodity'].str.lower().str.strip()

# Aligning names (e.g., FAO 'apple' -> commodity 'apple', FAO 'potato' -> 'potato')
print("FAO Crops Sample:", df_fao['crop_clean'].unique()[:10])
print("Price Commodities Sample:", df_prices_sample['Commodity_clean'].unique()[:10])

FAO Crops Sample: ['amaranth' 'apple' 'apricot' 'artichoke' 'asian pear' 'asparagus'
 'atemoya' 'avocado' 'babaco' 'banana - plantain']
Price Commodities Sample: ['dry chillies' 'tomato' 'beetroot' 'bhindi (ladies finger)'
 'bitter gourd' 'bottle gourd' 'brinjal' 'cabbage' 'carrot' 'cauliflower']


In [9]:
# Let's find overlapping crops between the FAO dataset and the price dataset
fao_crops = set(df_fao['crop_clean'].unique())
price_crops = set(df_prices_sample['Commodity_clean'].unique())
common_crops = fao_crops.intersection(price_crops)

print("Common crops found in both datasets:", common_crops)

# Select a valid common crop dynamically, fallback to the first common one or 'apple'
if common_crops:
    selected_crop_fao = list(common_crops)[0]
    selected_crop_price = selected_crop_fao
else:
    # Manual fallback setup if no exact string match exists
    selected_crop_fao = 'apple'
    selected_crop_price = 'apple'

# Filter datasets
crop_fao_info = df_fao[df_fao['crop_clean'] == selected_crop_fao].iloc[0]
storage_limit_days = int(crop_fao_info['storage_life_days_max']) if not pd.isna(crop_fao_info['storage_life_days_max']) else 30

print(f"\nSelected Crop: {selected_crop_fao.capitalize()}")
print(f"Maximum Storage Life: {storage_limit_days} days")

df_crop_prices = df_prices_sample[df_prices_sample['Commodity_clean'] == selected_crop_price].copy()
df_crop_prices['Arrival_Date'] = pd.to_datetime(df_crop_prices['Arrival_Date'])
df_crop_prices = df_crop_prices.sort_values(by='Arrival_Date')

# Drop duplicates for identical market dates
df_crop_prices = df_crop_prices.groupby(['Arrival_Date', 'State', 'District', 'Market']).agg({
    'Min_Price': 'mean',
    'Max_Price': 'mean',
    'Modal_Price': 'mean'
}).reset_index()

# Feature Engineering: 7-day and 30-day moving averages and volatility
df_crop_prices['MA_7'] = df_crop_prices.groupby(['State', 'Market'])['Modal_Price'].transform(lambda x: x.rolling(7, min_periods=1).mean())
df_crop_prices['MA_30'] = df_crop_prices.groupby(['State', 'Market'])['Modal_Price'].transform(lambda x: x.rolling(30, min_periods=1).mean())
df_crop_prices['Volatility_7'] = df_crop_prices.groupby(['State', 'Market'])['Modal_Price'].transform(lambda x: x.rolling(7, min_periods=1).std().fillna(0))

# Label Creation: If the price increases by > 10% within the storage window (e.g. 30 days), 'Store' (1), otherwise 'Sell' (0)
df_crop_prices['Future_Price'] = df_crop_prices.groupby(['State', 'Market'])['Modal_Price'].shift(-30)
df_crop_prices['Price_Diff_Pct'] = (df_crop_prices['Future_Price'] - df_crop_prices['Modal_Price']) / df_crop_prices['Modal_Price']
df_crop_prices['Decision'] = (df_crop_prices['Price_Diff_Pct'] > 0.10).astype(int) # 1 = Store, 0 = Sell

# Drop rows where we can't look ahead
df_ml = df_crop_prices.dropna(subset=['Future_Price', 'MA_7', 'MA_30', 'Volatility_7']).copy()

print("\nDecision Class Distribution:")
print(df_ml['Decision'].value_counts(normalize=True))
print(df_ml.head())

Common crops found in both datasets: {'lemon', 'pineapple', 'sweet potato', 'papaya', 'lime', 'spinach', 'turnip', 'garlic', 'cauliflower', 'cabbage', 'apple', 'orange', 'guava', 'basil', 'peach', 'mango', 'pomegranate', 'asparagus', 'plum', 'coconut', 'yam'}

Selected Crop: Lemon
Maximum Storage Life: 180 days

Decision Class Distribution:
Decision
0    0.654021
1    0.345979
Name: proportion, dtype: float64
  Arrival_Date        State     District                    Market  Min_Price  \
0   2019-01-01   Chandigarh   Chandigarh  Chandigarh (Grain/Fruit)     2000.0   
1   2019-01-01  Chattisgarh         Durg                      Durg     1500.0   
2   2019-01-01  Chattisgarh  Rajnandgaon               Rajnandgaon     4000.0   
3   2019-01-01      Gujarat    Ahmedabad                 Ahmedabad     1500.0   
4   2019-01-01      Gujarat    Ahmedabad             Dholka (Koth)     2000.0   

   Max_Price  Modal_Price    MA_7   MA_30  Volatility_7  Future_Price  \
0     4000.0       3000.0  

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Verify df_ml is defined and contains data
if 'df_ml' in globals() and not df_ml.empty:
    features = ['Min_Price', 'Max_Price', 'Modal_Price', 'MA_7', 'MA_30', 'Volatility_7']
    X = df_ml[features]
    y = df_ml['Decision']

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Train Classifier
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # Evaluation
    y_pred = model.predict(X_test)
    print("Model Evaluation Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Sell', 'Store']))
else:
    print("Dataframe 'df_ml' is not defined or empty. Please execute the previous cell successfully first.")

Model Evaluation Accuracy: 0.7987810920622337

Classification Report:
               precision    recall  f1-score   support

        Sell       0.83      0.88      0.85     13199
       Store       0.74      0.65      0.69      6983

    accuracy                           0.80     20182
   macro avg       0.78      0.76      0.77     20182
weighted avg       0.80      0.80      0.80     20182

